# HTTP semantics & idempotency, run

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 24 §24.1 · type: concept-lab*

**One-line promise:** read and reason about a real request/response, then prove to yourself
*which* operations are safe to retry by building a tiny idempotency-keyed handler that ignores
duplicate `POST`s — one resource, not two.

## 🧠 Why this matters

Everything an agentic system does crosses a network: the user's request, every model call,
every tool, every database write. HTTP is the contract that travels with each one — a **method**
(`GET`/`POST`/`PUT`/`DELETE`) that *means* something, a **path** that names a resource, headers,
an optional body, and a **status code** that tells the caller what happened. That contract isn't
decoration: clients, proxies, caches, and load balancers all make decisions based on it.

The single idea that ties HTTP to the rest of Part VII is **idempotency** — whether repeating a
call has the same effect as calling it once. Networks drop responses, so *every* call is
eventually retried by something. Get this right at the endpoint and a whole category of "why did
it charge twice / send two emails" bugs simply cannot happen. We build it here, fully in-memory,
no network, no key.

## Objectives & prereqs

**By the end you can:**
- Read a request as *method + path + headers + body* and a response as *status + headers + body*.
- Pick the right status code for an outcome (`201` create, `404` missing, `409` conflict, `422`
  validation) and justify the 4xx-vs-5xx split.
- Explain why `GET`/`PUT`/`DELETE` are idempotent by definition and bare `POST` is not.
- Make a create endpoint **safe under retry** with an idempotency key, so a duplicate `POST`
  yields one resource.

**Prereqs:** Python only; Ch 24 read. No API key, no network — the "server" is a dict-dispatch
stub in this process. (Foreshadows the idempotency keys of Ch 29 and the FastAPI service of Ch 25.)

In [ ]:
# --- Setup: imports, env, and the MOCK switch ---------------------------------
# stdlib only (+ python-dotenv from requirements.txt). No network is used at all:
# the "app" is an in-process callable, so MOCK is always on here.
import os
import json
import uuid
from dataclasses import dataclass, field

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass  # python-dotenv is optional for this fully-offline notebook

# This concept-lab has no live path — there is nothing to call. MOCK stays 1 so the
# notebook runs free and deterministically in CI and for readers without a key.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

# Deterministic ids so reruns produce identical output (uuid4 would not).
import random
random.seed(24)

def fake_id(prefix="msg"):
    return f"{prefix}_{random.getrandbits(32):08x}"

print(f"MOCK mode: {MOCK}  | in-process app, no sockets, no key required")

## 1 · A request and a response are just data (§24.1)

A client sends a **request** = `(method, path, headers, body)`. The server returns a
**response** = `(status, headers, body)`. Nothing more. Modeling them as plain dataclasses makes
the contract concrete and lets us route on it.

In [ ]:
@dataclass
class Request:
    method: str                    # GET / POST / PUT / DELETE ...
    path: str                      # /conversations/42/messages
    headers: dict = field(default_factory=dict)
    body: dict | None = None       # already-parsed JSON for this toy

@dataclass
class Response:
    status: int                    # 200, 201, 404, 409, 422 ...
    body: dict | None = None
    headers: dict = field(default_factory=dict)

    def __repr__(self):
        return f"Response(status={self.status}, body={json.dumps(self.body)})"

# One example of each. Methods carry meaning: GET reads, POST creates.
examples = [
    Request("GET",  "/messages"),
    Request("POST", "/messages", body={"text": "hello"}),
    Request("PUT",  "/messages/1", body={"text": "edited"}),
]
for r in examples:
    print(f"{r.method:<4} {r.path:<14} body={r.body}")

## 2 · The status-code table, as a contract (§24.1)

Status codes are how the server tells the client *what happened*, and infrastructure depends on
them. The split that matters most: **4xx is the caller's fault** ("don't retry blindly"); **5xx is
the server's fault** ("a retry might help").

In [ ]:
STATUS_TABLE = [
    (200, "2xx", "OK — request succeeded"),
    (201, "2xx", "Created — a new resource exists; return its id/Location"),
    (204, "2xx", "No Content — succeeded, nothing to return (often DELETE)"),
    (304, "3xx", "Not Modified — your cached copy is still good"),
    (400, "4xx", "Bad Request — malformed; CALLER's fault, don't retry"),
    (401, "4xx", "Unauthorized — not authenticated"),
    (403, "4xx", "Forbidden — authenticated but not allowed"),
    (404, "4xx", "Not Found — no such resource"),
    (409, "4xx", "Conflict — collides with current state (dup create)"),
    (422, "4xx", "Unprocessable — body parsed but failed validation"),
    (429, "4xx", "Too Many Requests — rate limited; back off"),
    (500, "5xx", "Internal Server Error — WE broke; a retry might help"),
    (503, "5xx", "Service Unavailable — upstream/overloaded"),
    (504, "5xx", "Gateway Timeout — upstream too slow"),
]
for code_, klass, meaning in STATUS_TABLE:
    print(f"{code_:>3} [{klass}]  {meaning}")

**What you just saw.** The class tells the caller how to react *without parsing the body*: a 4xx
says "fix your request — retrying unchanged is pointless"; a 5xx says "not your fault — a retry
may succeed." That distinction is the foundation of the retry logic in §11.4 and Ch 29.

## 3 · 🔧 A tiny 3-route app over an in-memory resource

Here is the "server": a dict-dispatch handler over a `messages` store. It implements `GET`
(read), `POST` (create), and `PUT` (replace) — and returns the *correct* status for each outcome,
including the error paths we'll predict next.

In [ ]:
# The entire "database": an in-memory dict of id -> message resource.
messages: dict[str, dict] = {}

def app(req: Request) -> Response:
    """A minimal router. Real frameworks (Ch 25, FastAPI) do this for you, but the
    method->meaning->status mapping is identical."""
    # ---- GET /messages/<id>  (read; idempotent, safe) ----
    if req.method == "GET":
        mid = req.path.rsplit("/", 1)[-1]
        if mid not in messages:
            return Response(404, {"error": "not found", "id": mid})
        return Response(200, messages[mid])

    # ---- POST /messages  (create; NOT idempotent on its own) ----
    if req.method == "POST":
        body = req.body or {}
        if "text" not in body:                       # validation gate
            return Response(422, {"error": "field 'text' is required"})
        mid = fake_id()
        messages[mid] = {"id": mid, "text": body["text"]}
        return Response(201, messages[mid], headers={"Location": f"/messages/{mid}"})

    # ---- PUT /messages/<id>  (replace; idempotent) ----
    if req.method == "PUT":
        mid = req.path.rsplit("/", 1)[-1]
        body = req.body or {}
        if "text" not in body:
            return Response(422, {"error": "field 'text' is required"})
        messages[mid] = {"id": mid, "text": body["text"]}   # full replace, same result every time
        return Response(200, messages[mid])

    return Response(405, {"error": f"method {req.method} not allowed"})

# A happy-path create, then read it back.
created = app(Request("POST", "/messages", body={"text": "first!"}))
print("POST ->", created, "| Location:", created.headers.get("Location"))
new_id = created.body["id"]
print("GET  ->", app(Request("GET", f"/messages/{new_id}")))

## 4 · 🔮 Predict the status codes

Before running the next cell, predict the **status** for each of these three requests:

1. `POST /messages` with body `{}` (no `text`) — ?
2. `GET /messages/does-not-exist` — ?
3. (we'll handle the *duplicate create → 409* case in §5)

Write your guesses down, then run it.

In [ ]:
bad_body = app(Request("POST", "/messages", body={}))
missing  = app(Request("GET", "/messages/does-not-exist"))

print("1) POST with no 'text' ->", bad_body.status, bad_body.body)
print("2) GET unknown id      ->", missing.status, missing.body)

assert bad_body.status == 422, "a parsed-but-invalid body is 422, not 400/500"
assert missing.status == 404
print("\nboth matched the contract.")

**What you just saw.** The empty body *parsed* fine but failed our rule, so it's **422
Unprocessable** — not `400` (which is for un-parseable/malformed requests) and definitely not
`500` (that would blame the server for the caller's mistake). The unknown id is **404**. Picking
the right code is what lets a client — or your own agent calling this API — react correctly.

## 5 · 🔧 Idempotency: make `POST` safe under retry

`GET`, `PUT`, and `DELETE` are idempotent *by definition* — repeating them lands the same final
state. Bare `POST` is not: two `POST`s create two resources. But networks drop responses, so the
client can't tell "it failed" from "it succeeded but I didn't hear back," and it **retries**.

The fix from the book: accept an **`Idempotency-Key`** header. The server records the key on first
success and, on a replay carrying the same key, returns the *original* result without creating
anything new. One key per *logical* operation — generated by the client before the first attempt,
reused across retries.

In [ ]:
# Key -> the response we returned the first time we saw it.
idempotency_store: dict[str, Response] = {}

def app_idempotent_post(req: Request) -> Response:
    """POST /messages that honors an Idempotency-Key header."""
    key = req.headers.get("Idempotency-Key")
    if key is not None and key in idempotency_store:
        # Replay: return the SAME result, create nothing. (Real systems often add
        # a header marking the response as replayed.)
        cached = idempotency_store[key]
        return Response(cached.status, cached.body,
                        headers={**cached.headers, "Idempotent-Replay": "true"})

    body = req.body or {}
    if "text" not in body:
        return Response(422, {"error": "field 'text' is required"})
    mid = fake_id()
    messages[mid] = {"id": mid, "text": body["text"]}
    resp = Response(201, messages[mid], headers={"Location": f"/messages/{mid}"})
    if key is not None:
        idempotency_store[key] = resp
    return resp

# A client generates ONE key for this logical create, then the network makes it retry.
client_key = "create:greeting:" + fake_id("op")
before = len(messages)

first  = app_idempotent_post(Request("POST", "/messages",
                                     headers={"Idempotency-Key": client_key},
                                     body={"text": "hello"}))
# ... response was dropped on the wire; the client retries the EXACT same request ...
second = app_idempotent_post(Request("POST", "/messages",
                                     headers={"Idempotency-Key": client_key},
                                     body={"text": "hello"}))

print("1st POST ->", first.status,  "id:", first.body["id"])
print("2nd POST ->", second.status, "id:", second.body["id"],
      "| replay:", second.headers.get("Idempotent-Replay"))
print("resources created by the two POSTs:", len(messages) - before)
assert first.body["id"] == second.body["id"], "replay must return the original resource"
assert len(messages) - before == 1, "two retries, ONE resource"
print("\nidempotency held: one resource, not two.")

**What you just saw.** The second `POST` carried the same `Idempotency-Key`, so the server
recognized the replay, returned the **original** `201` (same id) and created **nothing** new. The
client got a clean, correct answer even though the network ate the first response.

> ⚠️ **Pitfall — retrying a bare `POST`.** Wrap a non-idempotent create in a naive retry loop and
> a single dropped response becomes *two charges / two emails / two orders*. The cell below shows
> the damage with the **original** (non-idempotent) `app`, then the contrast.

In [ ]:
# The same dropped-response retry, but WITHOUT an idempotency key, against the naive app.
before = len(messages)
app(Request("POST", "/messages", body={"text": "charge $49.99"}))  # 1st attempt
app(Request("POST", "/messages", body={"text": "charge $49.99"}))  # retry: response was "lost"
print("naive POST retried once -> resources created:", len(messages) - before, "  <- duplicated!")

# The fix is structural, not a 'be careful': the key makes the duplicate impossible.
print("Idempotent POST retried -> would create: 1  (the server dedupes on the key)")

## 🎯 Senior lens

*Every* networked call is eventually retried by something — the client SDK, a proxy, a queue, an
impatient user mashing the button. So you don't get to assume "exactly once" delivery; you
**design** for it. The senior move is to make write endpoints retry-safe **on day one**: prefer
idempotent methods where the semantics allow (`PUT` over `POST` for "set this resource"), and for
genuine creates that must survive retries, bake in an idempotency key rather than hoping the
network behaves. It's one of those decisions that's nearly free up front and expensive to
retrofit once real clients depend on the endpoint — which is exactly the kind of trade-off Part VII
keeps returning to.

## Recap

- A request is `method + path + headers + body`; a response is `status + headers + body`. Methods
  are a **contract**, not a hint.
- Status codes drive client and infra behavior; the **4xx (caller's fault) vs 5xx (server's
  fault)** split decides whether a retry could ever help.
- Use the precise code: `201` create, `404` missing, `409` conflict, **`422`** for a parsed body
  that fails validation (not `400`/`500`).
- `GET`/`PUT`/`DELETE` are idempotent by definition; bare `POST` is not.
- An **idempotency key** makes a duplicate `POST` recognized and ignored — one resource, not two —
  the structural fix for retry-safety (returns in Ch 29).

## Exercises

Predict the result before running each.

1. **Add a `DELETE`.** Implement `DELETE /messages/<id>` returning `204` on success and `404` if
   the id is unknown. Then call it **twice** on the same id. Is `DELETE` idempotent? What status
   does the *second* call return, and is that a problem?
2. **Detect the conflict.** Add a `name` field that must be unique. Make `POST` return **`409`**
   when a message with that name already exists (instead of creating a duplicate). How does this
   differ from the idempotency-key path in §5?
3. **Key, not body.** In §5, change the *body* of the retried `POST` while keeping the same
   `Idempotency-Key`. Predict what should happen, then decide: should the server return the cached
   result, or reject the mismatch? Why might a real system store a hash of the body with the key?
4. **Per-attempt key (anti-pattern).** Regenerate the `Idempotency-Key` on the retry. Predict the
   resource count, run it, and explain why the key must be tied to the *logical operation*, not
   the attempt.

In [ ]:
# Exercise 1 — your code here

In [ ]:
# Exercise 2 — your code here

In [ ]:
# Exercise 3 — your code here

In [ ]:
# Exercise 4 — your code here

## Next

- ➡️ **Next notebook:** [`24-02-request-lifecycle-and-versioning.ipynb`](./24-02-request-lifecycle-and-versioning.ipynb)
  — trace one HTTPS request through DNS → TLS → load balancer → app, and evolve an API without
  breaking clients.
- 📘 **Template:** the HTTP/status-code/idempotency conventions you just built are baked into
  [`templates/fastapi-agent-service/`](../../../templates/fastapi-agent-service/), the starting
  point for Ch 25.
- 🏗️ **Capstone:** these are the contracts the capstone `app/` will honor — correct status codes
  and an idempotent `POST /runs` on the `/v1` surface.
- See the book **§24.1** for the status-code table and the idempotency key idea; idempotency keys
  return in full in **Ch 29**.